# An Ambiguous Arrangement of Dots

This notebook builds a controlled visual stimulus to study how spatial structure is perceived at fixation versus in peripheral vision.

## Perceptual Objective

The target is to generate an image where:
- the dot lattice is clearly visible near the fixation point,
- the same lattice becomes weak or unstable in the periphery.

This creates an illusion-like experience: dots appear where gaze lands, although the image itself is static.

## Why This Is Possible

Human visual encoding is not uniform across the retina:
- spatial acuity decreases with eccentricity,
- crowding and local pooling increase in the periphery,
- small chromatic differences are harder to segment away from fixation.

As a result, the same physical pattern can look structured at fixation and texture-like in peripheral vision.

<!-- TEASER_END -->

Let's first initialize the notebook:

In [ ]:
import numpy as np
np.set_printoptions(precision=6, suppress=True)
import os

## Method: Building The Stimulus Step By Step

### Step 1. Define image geometry
- Set image pixel dimensions `(N_height, N_width)`.
- Keep aspect ratio stable to preserve lattice geometry.

### Step 2. Build a hexagonal lattice
- Start from a regular Cartesian mesh.
- Shift alternate rows by half a column spacing.
- This row offset creates local hexagonal neighborhoods.

### Step 3. Define local shape primitives
- Draw a shape at each grid location: circle, diagonal, or polygon.
- Shape parameters modulate contour and orientation cues.

### Step 4. Control color in an interpretable way
- Convert hue/saturation to RGBA while approximately preserving luminance.
- This avoids strong luminance confounds in color manipulation.

### Step 5. Compose over a reference background
- Paint a background color first.
- Render all dots with controlled alpha and compositing.

### Step 6. Run one-parameter sweeps
- Scan one parameter at a time while keeping others fixed.
- Compare outputs to find the strongest fixation-dependent effect.

In [ ]:
%pip install pycairo

In [ ]:
import cairo
from IPython.display import Image, display
from math import pi
from io import BytesIO

DOWNSCALE = 8
N_height, N_width = int(np.sqrt(3)/2*12000)//DOWNSCALE, 12000//DOWNSCALE
N_height, N_width = int((1-.618)*12000)//DOWNSCALE, 12000//DOWNSCALE
N_height, N_width

In [ ]:
def disp(draw_func):
    surface = cairo.ImageSurface(cairo.FORMAT_ARGB32, N_width, N_height)
    ctx = cairo.Context(surface)
    draw_func(ctx, N_height=N_height, N_width=N_width)
    with BytesIO() as fileobj:
        surface.write_to_png(fileobj)
        display(Image(fileobj.getvalue(), width=N_width))

def render(draw_func, savepath):
    with cairo.PDFSurface(f"{savepath}.pdf", N_width, N_height) as surface:
        ctx = cairo.Context(surface)
        draw_func(ctx, N_height=N_height, N_width=N_width)        

figpath = '../files'
%mkdir -p {figpath}
import numpy as np
np.set_printoptions(precision=2, suppress=True)

import os
from IPython import get_ipython
ip = get_ipython()
# print(ip.user_ns)
study_name = None
if '__vsc_ipynb_file__' in ip.user_ns:
    # https://github.com/msm1089/ipynbname/issues/17
    study_name = os.path.split(os.path.basename(ip.user_ns['__vsc_ipynb_file__']))[1]
elif '__file__' in ip.user_ns:
    study_name = ip.user_ns['__file__']#.replace('.ipynb', '')
else:
    import ipynbname
    study_name = ipynbname.name()

savepath = os.path.join(figpath, study_name.replace('.ipynb', ''))
savepath

In [ ]:
def hue_to_rgba(hue, sat=1., alpha=1., chroma_max = 0.2):
    """
    Convert a hue value to an isoluminant RGB color.
    The hue value should be in degrees [0, 360).
    sat in [0, 1] controls chroma while keeping luminance constant.
    """
    hue = np.mod(hue, 360.)
    theta = np.deg2rad(hue)
    sat = float(np.clip(sat, 0.0, 1.0))

    # Linear-RGB luminance weights.
    luma = np.array([0.2126, 0.7152, 0.0722])

    # Two orthonormal directions in the plane orthogonal to luminance.
    basis_u = np.array([luma[1], -luma[0], 0.0])
    basis_u /= np.linalg.norm(basis_u)
    basis_v = np.cross(luma, basis_u)
    basis_v /= np.linalg.norm(basis_v)

    # Constant luminance in linear RGB, with a safe chroma radius.
    base_luminance = 0.5
    5
    rgb_linear = base_luminance + sat * chroma_max * (
        np.cos(theta) * basis_u + np.sin(theta) * basis_v
    )
    rgb_linear = np.clip(rgb_linear, 0.0, 1.0)

    return (*rgb_linear, alpha)

In [ ]:
def do_shape(cr, x, y, radius, shape_mode='circle', n_sides=6, angle0=0.0):
    """
    Draw either a circle or a regular polygon centered at (x, y).

    shape_mode: 'circle', 'polygon', or 'diagonal'
    n_sides: number of polygon sides when shape_mode='polygon'
    """
    rr = radius / 2.0

    if shape_mode == 'diagonal':
        # Diagonal mode = rotated square (diamond-like look).
        n_sides = 4
        angle0 = pi / 4

    if shape_mode in ('polygon', 'diagonal'):
        n_sides = int(max(3, n_sides))
        cr.new_sub_path()
        for k in range(n_sides):
            angle = angle0 + 2.0 * pi * k / n_sides
            px = x + rr * np.cos(angle)
            py = y + rr * np.sin(angle)
            if k == 0:
                cr.move_to(px, py)
            else:
                cr.line_to(px, py)
        cr.close_path()
    else:
        cr.save()
        cr.translate(x, y)
        cr.scale(rr, rr)
        cr.arc(0.0, 0.0, 1.0, 0.0, 2.0 * pi)
        cr.restore()


In [ ]:
def hexagonal_grid(cr, N_height, N_width, N_H, N_W, size_mag, alpha, c_mean, c_std, s_mean, s_std, operator, shape_mode='diagonal', n_sides=6):

    cr.save()
    cr.set_operator(cairo.OPERATOR_SOURCE)
    cr.set_source_rgba(*hue_to_rgba(c_mean, sat=s_mean, alpha=1.))
    cr.paint()
    cr.restore()

    cr.set_operator(operator)

    # Compute the grid
    # https://laurentperrinet.github.io/sciblog/posts/2020-04-16-creating-an-hexagonal-grid.html
    width_v, height_v = np.meshgrid(np.linspace(0, N_width, N_W+2, endpoint=True)[1:-1],
                                    np.linspace(0, N_height, N_H+2, endpoint=True)[1:-1], sparse=False, indexing='xy')

    width_v[::2, :] += N_width/N_W/4  # shift every second row by half a column
    width_v[1::2, :] -= N_width/N_W/4  # shift every second row by half a column

    # convert to cartesian coordinates
    X = width_v
    Y = height_v
    R = size_mag * N_height / N_H * np.ones_like(X)  # constant radius
    C = (c_mean + c_std) * np.ones_like(X)  # incremented hue values
    S = s_mean - s_std * np.ones_like(X)  # saturation values

    # draw
    for x, y, r, c, s in zip(X.ravel(), Y.ravel(), R.ravel(), C.ravel(), S.ravel()):
        do_shape(cr, x, y, r, shape_mode=shape_mode, n_sides=n_sides)
        cr.set_source_rgba(*hue_to_rgba(c, sat=s, alpha=alpha))
        cr.fill()

    return cr


## results



In [ ]:
N = 5
N = 8
N = 24
N = 35
N = 71
N = 20
N_H, N_W = int(N*N_height/N_width), N
N*N_height/N_width, N_H, N_W

In [ ]:

# blue
c_ref = 180
c_ref = 0
c_std = 8
s_ref = .7
s_std = .3
opts = dict(N_height=N_height, N_width=N_width, N_H=N_H, N_W=N_W,
            size_mag=.2, alpha=1., c_mean=c_ref, c_std=c_std, s_mean=s_ref, s_std=s_std,
            operator=cairo.OPERATOR_OVER, shape_mode='polygon', n_sides=6)

@disp
def draw(cr, N_height=N_height, N_width=N_width): cr = hexagonal_grid(cr, **opts)


### Baseline Stimulus Interpretation

**Why this cell exists**
- It defines the reference stimulus before systematic parameter sweeps.
- It provides the visual baseline for all comparisons below.

**What to evaluate visually**
- Dot segmentation at fixation.
- Loss of structure in the periphery.
- Overall balance between visibility and ambiguity.

**Interpretation rule**
- Too visible everywhere: effect is too strong globally.
- Invisible everywhere: effect is too weak.
- Best regime: sharp near fixation, weaker in peripheral regions.

In [ ]:
def draw(cr, N_height=N_height, N_width=N_width): cr = hexagonal_grid(cr, **opts)
render(draw, savepath)

# scanning some parameters

In [ ]:
N_scan = 9

## alpha

In [ ]:
opts_ = opts.copy()
for alpha_ in np.linspace(0.1, 1., N_scan, endpoint=True): 
    opts_.update(alpha=alpha_)
    print(f'{alpha_=:.2e}')
    @disp
    def draw(cr, N_height=N_height, N_width=N_width): cr = hexagonal_grid(cr, **opts_)


### Scan Analysis: Alpha (Opacity)

**Why scan this parameter**
- Alpha controls dot-to-background contrast strength.
- It is a direct threshold control for detectability.

**Visual analysis**
- Low alpha tends to flatten the whole field.
- High alpha makes dots salient everywhere and weakens the illusion.
- Intermediate alpha usually maximizes fixation-dependent visibility.

## size_mag

In [ ]:
opts_ = opts.copy()
for size_mag_ in np.geomspace(0.02, .5, N_scan, endpoint=True): 
    opts_.update(size_mag=size_mag_)
    print(f'{size_mag_=:.2e}')
    @disp
    def draw(cr, N_height=N_height, N_width=N_width): cr = hexagonal_grid(cr, **opts_)


### Scan Analysis: size_mag (Dot Size)

**Why scan this parameter**
- Dot size sets local support relative to grid spacing.
- It strongly controls peripheral resolvability.

**Visual analysis**
- Very small dots are near threshold and can fade globally.
- Large dots are easy to see across the image.
- The strongest illusion is usually in the small-to-medium range.

## c_mean

In [ ]:
opts_ = opts.copy()
for c_mean_ in np.linspace(0., 360., N_scan, endpoint=False): 
    opts_.update(c_mean=c_mean_)
    print(f'{c_mean_=:.2f}')
    @disp
    def draw(cr, N_height=N_height, N_width=N_width): cr = hexagonal_grid(cr, **opts_)


### Scan Analysis: c_mean (Mean Hue)

**Why scan this parameter**
- Mean hue changes chromatic alignment with the background.
- Some hue neighborhoods are more separable than others.

**Visual analysis**
- Certain hues increase dot/background separation and clarity.
- Others reduce segmentation and increase texture-like appearance.
- Keep hue regions that preserve central visibility without global over-visibility.

## c_std


In [ ]:
opts_ = opts.copy()
for c_std_ in np.linspace(0., 30, N_scan, endpoint=False): 
    opts_.update(c_std=c_std_)
    print(f'{c_std_=:.2e}')
    @disp
    def draw(cr, N_height=N_height, N_width=N_width): cr = hexagonal_grid(cr, **opts_)


### Scan Analysis: c_std (Hue Spread)

**Why scan this parameter**
- Hue spread controls chromatic variability across dots.
- It modulates texture heterogeneity and grouping cues.

**Visual analysis**
- Low spread produces a more uniform and often flatter field.
- High spread increases local differences and global salience.
- A moderate spread often gives the best fixation-periphery contrast.

## s_mean

In [ ]:
opts_ = opts.copy()
# for s_mean_ in np.linspace(0., s_ref, N_scan, endpoint=True): 
for s_mean_ in np.linspace(s_std, 1., N_scan, endpoint=True): 
    opts_.update(s_mean=s_mean_)
    print(f'{s_mean_=:.2f}')
    @disp
    def draw(cr, N_height=N_height, N_width=N_width): cr = hexagonal_grid(cr, **opts_)


### Scan Analysis: s_mean (Mean Saturation)

**Why scan this parameter**
- Mean saturation controls chromatic intensity at fixed geometry.
- It changes the strength of color-defined edges.

**Visual analysis**
- Low saturation weakens segmentation cues.
- High saturation can make dots stable everywhere.
- Intermediate saturation is typically best for the illusion.

## s_std


In [ ]:
opts_ = opts.copy()
# for s_std_ in np.linspace(0., 1.-s_ref, N_scan, endpoint=True): 
for s_std_ in np.linspace(0., s_ref, N_scan, endpoint=True): 
    opts_.update(s_mean=s_ref, s_std=s_std_)
    print(f'{s_std_=:.2e}')
    @disp
    def draw(cr, N_height=N_height, N_width=N_width): cr = hexagonal_grid(cr, **opts_)


### Scan Analysis: s_std (Saturation Spread)

**Why scan this parameter**
- Saturation spread introduces local contrast variability.
- It can amplify or dampen local textural anchors.

**Visual analysis**
- Very low spread yields smoother appearance.
- High spread creates stronger local fluctuations.
- Moderate spread often preserves ambiguity in the periphery while keeping central structure readable.

## N_H

In [ ]:
opts_ = opts.copy()
N_Hs = np.unique([int(k) for k in N_H * np.logspace(-1, 1, N_scan, base=2, endpoint=True)]).astype(int)
for N_H_ in N_Hs: 
    opts_.update(N_H=N_H_,N_W=N_H_)
    print(f'{N_H_=}')
    @disp
    def draw(cr, N_height=N_height, N_width=N_width): cr = hexagonal_grid(cr, **opts_)

### Scan Analysis: N_H and N_W (Grid Density)

**Why scan this parameter**
- Grid density sets the effective spatial frequency of the lattice.
- Peripheral encoding is very sensitive to this spacing scale.

**Visual analysis**
- Sparse lattices are easy to parse across the field.
- Dense lattices increase peripheral fusion and can strengthen fixation dependence.
- The useful regime is close to the central/peripheral segmentation boundary.

## N_width

In [ ]:
# opts_ = opts.copy()
# N_widths = [int(k) for k in N_width * np.logspace(-1, 1, N_scan, base=1.5, endpoint=True)]
# for N_width_ in N_widths: 
#     opts_.update(N_width=N_width_)
#     print(f'{N_width_=}')
#     @disp
#     def draw(cr, N_height=N_height, N_width=N_width): cr = hexagonal_grid(cr, **opts_)

## `cairo.OPERATOR_*`

In [ ]:
# opts_ = opts.copy()
# for operator_ in [cairo.OPERATOR_CLEAR, cairo.OPERATOR_ADD, cairo.OPERATOR_SOURCE, cairo.OPERATOR_OVER, cairo.OPERATOR_OUT, cairo.OPERATOR_SCREEN, cairo.OPERATOR_DEST_OUT]: 
#     opts_.update(operator=operator_)
#     print(f'{operator_}')
#     @disp
#     def draw(cr, N_height=N_height, N_width=N_width): cr = hexagonal_grid(cr, **opts_)


## shape exploration

In [ ]:
# Exploration of different shape families
opts_ = opts.copy()
opts_.update(size_mag=.5, N_H=max(6, N_H//2), N_W=max(6, N_W//2))

shape_configs = [
    ('circle', 6),
    ('diagonal', 4),
    ('polygon', 3),
    ('polygon', 4),
    ('polygon', 5),
    ('polygon', 6),
    ('polygon', 8),
]

for shape_mode_, n_sides_ in shape_configs:
    opts_.update(shape_mode=shape_mode_, n_sides=n_sides_)
    print(f'shape_mode={shape_mode_}, n_sides={n_sides_}')

    @disp
    def draw(cr, N_height=N_height, N_width=N_width):
        cr = hexagonal_grid(cr, **opts_)


### Scan Analysis: Shape Family

**Why scan shape families**
- Different shapes provide different corner and orientation cues.
- Shape class changes local grouping at constant color and density.

**Visual analysis**
- Circle yields smoother contours and weaker orientation cues.
- Diagonal and low-side polygons provide sharper anchors.
- Use smoother shapes for stronger ambiguity, and angular shapes for stronger local snapping at fixation.

## some book keeping for the notebook

In [ ]:
%pwd

In [ ]:
%load_ext watermark
%watermark -i -h -m -v -p numpy,cairo  -r -g -b